In [1]:
import numpy as np
import pandas as pd, os, datetime

import matplotlib.pyplot as plt

# Import the loader function
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
from process_code import load_generation_data

Dask dashboard: /proxy/8787/status


2026-06-26 12:33:28,728 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 3bed0ad80cfb8170b39164e0292167f3 initialized by task ('shuffle-transfer-3bed0ad80cfb8170b39164e0292167f3', 370) executed on worker tcp://127.0.0.1:37561
2026-06-26 12:33:50,052 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 3bed0ad80cfb8170b39164e0292167f3 deactivated due to stimulus 'task-finished-1782441230.0063052'


In [2]:
data, info = load_generation_data(
    sdate="2009-07-01",
    edate="2024-06-30",
    mode="daily",
    ftype=["Wind"],
    apply_remove_negatives=True,
    apply_remove_wind_zeros=True,
    apply_min_heatwave_days=True,
    min_heatwave_days_threshold=20,
    apply_clear_agc=False
)

Read gen_details & hw_tseries with Dask: 0.27 sec
Select group: 0.03 sec
--- Starting Dask-Native Process ---


/g/data/ng72/ms5578/ID_HW_BARRA/process_code.py:168: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grp_pd_jittered = grp_pd.groupby(['lat', 'lon'], group_keys=False).apply(


Starting final Dask compute...
Dask compute finished.

--- DASK TIMING REPORT ---
duid_setup: 0.00 seconds
csv_bulk_read: 1.99 seconds
filter_and_clean: 0.02 seconds
type_conversion_and_dropna: 0.07 seconds
date_filter: 0.02 seconds
hw_tseries_filter: 0.04 seconds
aggregate_daily: 0.12 seconds
jitter: 1.04 seconds
final_merge: 0.07 seconds
compute: 40.38 seconds
--- END REPORT ---

Process group: 49.49 sec

--- DEBUG: Calculated Heatwave Days per Generator ---
DUID
MEWF1       236
COOPGWF1    148
WRWF1       146
SAPHWF1     129
MUSSELR1    112
BODWF1       91
GRANWF1      84
MTGELWF1     60
STWF1        58
BALDHWF1     57
TARALGA1     54
LKBONNY2     52
KABANWF1     52
WOODLWN1     50
HDWF2        48
BOCORWF1     48
HDWF3        48
HDWF1        48
CROOKWF2     48
LKBONNY3     47
LGAPWF1      45
GUNNING1     45
GULLRWF1     45
NBHWF1       42
DULAWF1      41
HALLWF2      41
HALLWF1      41
CROWLWF1     40
CRURWF1      39
KIATAWF1     38
CLEMGPWF     38
MACARTH1     37
WATERLWF     36
WG

In [3]:
df = data.copy()

df["year"] = pd.to_datetime(df["time"]).dt.year
first_year = (
    df.groupby("DUID", as_index=False)["year"]
      .min()
      .rename(columns={"year": "first_year"})
)

print(first_year)

        DUID  first_year
0      ARWF1        2016
1   BALDHWF1        2015
2     BLUFF1        2011
3   BOCORWF1        2014
4     BODWF1        2018
5   BULGANA1        2020
6    CHYTWF1        2020
7   CLEMGPWF        2009
8   CNUNDAWF        2021
9   COOPGWF1        2019
10  CROOKWF2        2018
11  CROWLWF1        2018
12   CRURWF1        2020
13   CTHLWF1        2020
14   DULAWF1        2023
15  FLYCRKWF        2023
16   GRANWF1        2019
17  GULLRWF1        2013
18  GUNNING1        2011
19   HALLWF1        2009
20   HALLWF2        2009
21     HDWF1        2016
22     HDWF2        2017
23     HDWF3        2017
24  KABANWF1        2022
25    KEPWF1        2021
26  KIATAWF1        2017
27   LGAPWF1        2019
28   LGAPWF2        2021
29  LKBONNY1        2021
30  LKBONNY2        2009
31  LKBONNY3        2010
32  MACARTH1        2012
33  MERCER01        2013
34     MEWF1        2018
35  MTGELWF1        2018
36  MUSSELR1        2013
37   MUWAWF1        2019
38    NBHWF1        2010


In [19]:
counts = (
    df.groupby("DUID")["EHF_flag"]
      .value_counts()
      .unstack(fill_value=0)
      .rename(columns={0: "baseline", 1: "heatwave"})
      .reset_index()
)

print(counts)

EHF_flag      DUID  baseline  heatwave
0            ARWF1      1797        27
1         BALDHWF1      1770        57
2           BLUFF1      1803        24
3         BOCORWF1      1779        48
4           BODWF1      1735        91
5         BULGANA1      1472        25
6          CHYTWF1      1490        35
7         CLEMGPWF      1783        38
8         CNUNDAWF       912        23
9         COOPGWF1      1679       148
10        CROOKWF2      1779        48
11        CROWLWF1      1782        40
12         CRURWF1      1276        39
13         CTHLWF1      1612        25
14         DULAWF1       413        41
15        FLYCRKWF       304        24
16         GRANWF1      1588        84
17        GULLRWF1      1782        45
18        GUNNING1      1780        45
19         HALLWF1      1782        41
20         HALLWF2      1785        41
21           HDWF1      1778        48
22           HDWF2      1779        48
23           HDWF3      1777        48
24        KABANWF1       

In [20]:
details = first_year.merge(counts[['DUID','heatwave','baseline']], on='DUID')

In [21]:
details = details.merge(info[['station_name','region','DUID']], on='DUID')

In [22]:
details.to_csv("data/output/wind_chapter/wind_details_table.csv")

In [23]:
base = info[["DUID","region","lat","lon"]]
base

,DUID,region,lat,lon
0,ARWF1,VIC1,-37.238518,143.079431
1,BALDHWF1,VIC1,-38.790800,145.927700
4,BLUFF1,SA1,-33.369331,138.982476
6,BOCORWF1,NSW1,-36.576955,149.125867
7,BODWF1,NSW1,-32.414600,149.099795
11,BULGANA1,VIC1,-37.118103,142.960530
17,CHYTWF1,VIC1,-37.099382,145.267835
18,CLEMGPWF,SA1,-33.508568,138.119175
19,CNUNDAWF,SA1,-37.760253,140.403311
22,COOPGWF1,QLD1,-26.733227,151.472270


In [24]:
clustered_info = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/wind_spatial_clusters.csv")
profs = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/wind_profile_clusters.csv")
loc_df = clustered_info[["DUID", "lat","lon","cluster_name"]]
loc_df = pd.merge(loc_df,profs[["DUID","profile_name"]],on='DUID')
loc_df = pd.merge(loc_df,info[["DUID","region"]],on='DUID')
loc_df = pd.merge(loc_df,details[["DUID","first_year","heatwave","baseline"]],on='DUID')
loc_df

,DUID,lat,lon,cluster_name,profile_name,region,first_year,heatwave,baseline
0,ARWF1,-37.238518,143.079431,West Victoria,Minimal change,VIC1,2016,27,1797
1,BALDHWF1,-38.790800,145.927700,Bass Strait,Minimal change,VIC1,2015,57,1770
2,BLUFF1,-33.369331,138.982476,South Australia,Minimal change,SA1,2011,24,1803
3,BOCORWF1,-36.576955,149.125867,Alpine NSW,Minimal change,NSW1,2014,48,1779
4,BODWF1,-32.414600,149.099795,Alpine NSW,Minimal change,NSW1,2018,91,1735
5,BULGANA1,-37.118103,142.960530,West Victoria,Daytime increase,VIC1,2020,25,1472
6,CHYTWF1,-37.099382,145.267835,West Victoria,Minimal change,VIC1,2020,35,1490
7,CLEMGPWF,-33.508568,138.119175,South Australia,Morning peak,SA1,2009,38,1783
8,CNUNDAWF,-37.760253,140.403311,West Victoria,Daytime increase,SA1,2021,23,912
9,COOPGWF1,-26.733227,151.472270,Eastern Central,All-day reduction,QLD1,2019,148,1679


In [25]:
loc_df.to_csv("data/output/wind_chapter/wind_loc_table.csv")